In [22]:
import pandas as pd
from uuid import uuid4
from transformers import AutoTokenizer
from math import ceil
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict
from tqdm import tqdm


tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2" ## adjust tokenization model to the one that is used in the embedding/retriever achitecture # "McGill-NLP/LLM2Vec-Meta-Llama-31-8B-Instruct-mntp"
)

token_length = 256 # adjust to maximal token length
document_limit = 1000
dataset = 'train' # test

# restricting legth (make room for cls token and paragraph seperators)
TOK_LEN = token_length - 10

In [2]:
# read  data
df = pd.read_parquet(f'/raid/deallab/SF_RAG_Data/ASQA/{dataset}.parquet')

for col in df.columns:
    print(col,':')
    print(df.loc[1, col], '\n')

ambiguous_question :
Who won the 2016 ncaa football national championship? 

qa_pairs :
[{'context': "The 13–1 Alabama Crimson Tide won the game, holding off the undefeated Clemson Tigers 45–40 in the fourth quarter. Accompanied by a talented receiving corps, Clemson's Heisman Finalist quarterback Deshaun Watson had a historic performance, setting the record for most total yards in national championship game history, with 478 yards (405 passing / 73 rushing) against the nation's third-ranked defense in Alabama, breaking the record previously set by Vince Young in the 2006 Rose Bowl. Following the game, the AP Poll also named Alabama as its top team of the season, giving Alabama their fourth title in seven seasons. Both Clemson and Alabama finished the season 14–1.", 'question': "Who won the 2016 season's ncaa football national championship?", 'short_answers': array(['Clemson Tigers', '2016 Clemson Tigers football team',
        '2016 Clemson Tigers football', 'the Tigers', 'Clemson',
 

In [3]:
#parse tables to text
def get_table(table):
    table_text = []
    for i, tr in enumerate(table.find('tbody').findChildren("tr" , recursive=False)):
        tr_text = tr.get_text()
        tr_text = re.sub(r'\n+',';',tr_text).strip(';')
        if not tr_text: continue
        if table_text == []:
            tr_text = '\n#### Table: ' + tr_text
        table_text.append(tr_text)

    return '\n'.join(table_text)

# parse pars to text
def get_p(par):
    p_text = par.get_text()
    p_text = p_text.replace('\n','')
    return p_text

def get_h(heading):
    h = heading.find(['h1', 'h2', 'h3','h4', 'h5'])
    try:
        heading_type = int(re.search(r'<h(\d)', str(h)).group(1))
    except:
        print(heading)
        raise
    h_text ='\n' + ' '.join(['#'*heading_type,h.get_text()])
    return h_text

#pars unordered lists to text
def get_ul(ul):
    list_text = []
    for li in ul.find_all('li'):
        list_text.append('* ' + li.get_text())
    return '\n'.join(list_text)

# pars ordered list to text
def get_ol(ol):
    list_text = []
    for i, li in enumerate(ol.find_all('li')):
        if li.get_text():
            list_text.append(' '.join([str(i+1),li.get_text().replace('\n','')]))
    return '\n'.join(list_text)
        

# parse whole document
def parse_document(doc):
    content = doc.find_all(['div', 'p', 'table', 'ul', 'ol'])
    document  = []
    for cont in content:
        #stop condition
        if cont.name == 'div' and cont.find(['h1', 'h2', 'h3','h4', 'h5'], id=['See_also', 'References']):
            break
        
        #get headining
        if cont.name == 'div' and cont.has_attr('class') and  'mw-heading' in cont['class']:
            document.append(get_h(cont))
        # get par
        elif cont.name == 'p':
            par = get_p(cont)
            if par:
                document.append(par)
        #get ul
        elif cont.name == 'ul':
            document.append(get_ul(cont))
        #get ol
        elif cont.name == 'ol':
            document.append(get_ol(cont))
        #get table
        elif cont.name == 'table':
            if cont.has_attr('class') and 'metadata' in cont['class']: continue
            document.append(get_table(cont))
        # explore div
        elif cont.name == 'div':
            document.append(parse_document(cont))

    return '\n'.join(document).strip('\n')



In [13]:
# create embedding document dataset.
evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'question', 'text'])

#fetched_documents = {}
for idx, row in tqdm(df.iterrows(), total=min(document_limit, len(df))):
    if idx == document_limit: break # change number of document to chunk/process
    sample_id = row['sample_id']
    evidences = row['wikipages']
    q1 = row['ambiguous_question']
    q2 = defaultdict(list)
    for q in row['qa_pairs']:
        question = q['question']
        wikipage = q['wikipage']
        if not question or not wikipage: continue
        q2[wikipage].append(question)
    for evidence in evidences:
        url = evidence['url']
        #if url in fetched_documents: continue
        #fetched_documents[url] = []
        page = requests.get(url)
        
        # Create a BeautifulSoup object
        soup = BeautifulSoup(page.text, 'html.parser')
        # get title
        title = soup.find(id='firstHeading').get_text()
        
        #extract content
        content = soup.find(class_='mw-content-ltr')
        parsed_doc = parse_document(content)
        
        # chunk document
        documents = [[]]
        
        for par in re.split(r'(?=\n#{1,4})', parsed_doc):
            tokenized_par = tokenizer.encode(par, add_special_tokens = False)
            length = len(tokenized_par)
            if len(documents[-1]) + length < TOK_LEN:
                documents[-1].extend(tokenized_par)
            elif length > TOK_LEN:
                begin = 0 
                while begin < length:
                    if begin + TOK_LEN >= length:
                        documents.append(tokenized_par[begin:])
                        break
                    documents.append(tokenized_par[begin:begin + TOK_LEN])
                    begin += TOK_LEN - int(TOK_LEN * 0.1)
            else:
                documents.append(tokenized_par)
            
        # print(len(documents))
        for doc in documents:
            doc_text = tokenizer.decode(doc)
            id = uuid4()
            #fetched_documents[url].append(id)
            evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, q1, doc_text]
        
        if title in q2:
            for question in q2[title]:
                for doc in documents:
                    doc_text = tokenizer.decode(doc)
                    id = uuid4()
                    #fetched_documents[url].append(id)
                    evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, question, doc_text]
                
#print(fetched_documents)
evidence_df.to_csv('/raid/deallab/SF_RAG_Data/ASQA/embedding_train.csv', index=False)
evidence_df.head()

KeyboardInterrupt: 

In [7]:
#creating question question pairs for retrival training (not implemented)
qq_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions'])

# add question follow up questions to df
for idx, row in tqdm(df.iterrows(), total=min(document_limit, len(df))):
    if idx == document_limit: break # change number of document to chunk/process
    sample_id = row['sample_id']
    question = row['ambiguous_question']
    follow_up_question = '\n'.join([f'### {q["question"]}' for q in row['qa_pairs']])
    qq_df.loc[len(qq_df)] = [uuid4(), sample_id,  question, follow_up_question]
    
evidence_df.to_csv(f'/raid/deallab/SF_RAG_Data/ASQA/follow_up_train.csv', index=False)
qq_df

,id,sample_id,question,follow_up_questions
0,074867be-6353-4365-8847-d8ed4bc28b8b,-5742327688291876861,When does the new bunk'd come out?,### When does episode 42 of bunk'd come out?\n...
1,aae9bcbf-87b7-4a12-bff7-a1210bc63f3e,-3582047784487750233,Who won the 2016 ncaa football national champi...,### Who won the 2016 season's ncaa football na...
2,b44ea641-a005-4dc9-a2c1-5a45662d7172,6811938153834854976,When was the last time the death penalty was u...,"### As of 2017, when was the last time the dea..."
3,71c9f0db-601e-4cca-bd7d-be24a975165a,1700733897006170137,Where will failure of the left ventricle cause...,### Where does failure of the left ventricle c...
4,a213c9a9-5e6b-40ac-82a2-2fa217cd4415,142117929623619257,Who won the war between ethiopia and italy?,### Who won the First Italo-Ethiopian War?\n##...
...,...,...,...,...
995,18913ec3-9343-4df0-92c5-e0ef5a31381b,7635846020280787536,When did jersey shore family vacation episode ...,"### When did Season 1, episode 3 of Jersey Sho..."
996,defc5a02-1042-4358-aaf4-d0a563e6e8df,4520201763018144934,How old do u have to be to get a tattoo in was...,### How old do you have to be to get a tattoo ...
997,13c2dcc5-f621-45e8-82bd-38a9e2cec771,-2226177198374409130,Who won the national championship college foot...,### Who won the national championship game for...
998,8ed41759-eabc-4048-9731-ecc456a91eff,-4657262659143356228,Who was the beast in beauty and the beast tv s...,### What was the name of the beast in the 1987...


In [14]:
# read  data
df = pd.read_parquet(f'/raid/deallab/SF_RAG_Data/ASQA/dev.parquet')

for col in df.columns:
    print(col,':')
    print(df.loc[1, col], '\n')

ambiguous_question :
Who is the original artist of sound of silence? 

qa_pairs :
[{'context': 'Sounds of Silence is the second studio album by Simon & Garfunkel, released on January 17, 1966. The album\'s title is a slight modification of the title of the duo\'s first major hit, "The Sound of Silence", which originally was released as "The Sounds of Silence". The song had earlier been released in an acoustic version on the album "Wednesday Morning, 3 A.M.", and later on the soundtrack to the movie "The Graduate". Without the knowledge of Paul Simon or Art Garfunkel, electric guitars, bass and drums were overdubbed by Columbia Records staff producer Tom Wilson on June 15, 1965. This new version was released as a single in September 1965, and opens the album.', 'question': 'Who is the original artist of sound of silence, the song, released in 1964?', 'short_answers': array(['Simon & Garfunkel', 'Paul Simon and Art Garfunkel',
        'Art Garfunkel', 'Paul Simon'], dtype=object), 'wikip

In [23]:
# create embedding document dataset.
evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'text'])
qa_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions', 'long_answers', 'short_answers'])

fetched_documents = []
for idx, row in tqdm(df.iterrows(), total=min(document_limit, len(df))):
    if idx == document_limit: break # change number of document to chunk/process
    sample_id = row['sample_id']
    evidences = row['wikipages']
    question = row['ambiguous_question']
    follow_up_question = [q["question"] for q in row['qa_pairs']]
    long_answers = [ann['long_answer'] for ann in row['annotations']]
    short_answers = [list(ann['short_answers']) for ann in row['qa_pairs']]
    for evidence in evidences:
        url = evidence['url']
        if url in fetched_documents: continue
        fetched_documents.append(url)
        page = requests.get(url)
        
        # Create a BeautifulSoup object
        soup = BeautifulSoup(page.text, 'html.parser')
        # get title
        title = soup.find(id='firstHeading').get_text()
        
        #extract content
        content = soup.find(class_='mw-content-ltr')
        parsed_doc = parse_document(content)
        
        # chunk document
        documents = [[]]
        
        for par in re.split(r'(?=\n#{1,4})', parsed_doc):
            tokenized_par = tokenizer.encode(par, add_special_tokens = False)
            length = len(tokenized_par)
            if len(documents[-1]) + length < TOK_LEN:
                documents[-1].extend(tokenized_par)
            elif length > TOK_LEN:
                begin = 0 
                while begin < length:
                    if begin + TOK_LEN >= length:
                        documents.append(tokenized_par[begin:])
                        break
                    documents.append(tokenized_par[begin:begin + TOK_LEN])
                    begin += TOK_LEN - int(TOK_LEN * 0.1)
            else:
                documents.append(tokenized_par)
            
        # print(len(documents))
        for doc in documents:
            doc_text = tokenizer.decode(doc)
            id = uuid4()
            #fetched_documents[url].append(id)
            evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, doc_text]
        
    qa_df.loc[len(qa_df)] = [uuid4(), sample_id, question, follow_up_question, long_answers, short_answers]
                
#print(fetched_documents)
evidence_df.to_csv('/raid/deallab/SF_RAG_Data/ASQA/evidence_test.csv', index=False)
qa_df.to_csv('/raid/deallab/SF_RAG_Data/ASQA/qa_test.csv', index=False)
print(evidence_df.head())
print(qa_df.head())

  0%|          | 0/948 [00:02<?, ?it/s]


KeyboardInterrupt: 